# <center>**03 - Train**<center>

Notebook huấn luyện một trong bốn kiến trúc trong `models.py` (`small`, `medium`, `medium_01`, `large`) trên dữ liệu đã xử lý ở Ngày 1.

**Luồng chạy:**

1. Import thư viện, thiết lập đường dẫn và logging ghi ra console và thư mục `logs/`.
2. Cấu hình kiến trúc, số epoch, learning rate và chế độ chạy thử.
3. Đọc dữ liệu đã xử lý, tạo `Dataset` và `DataLoader`.
4. Khởi tạo mô hình CNN.
5. Huấn luyện bằng hàm `fit()` trong `train_utils.py`, lưu checkpoint tốt nhất và ghi lịch sử từng epoch ra CSV.

## **1. Import, đường dẫn và logging**

Chuẩn bị môi trường Python, thư viện huấn luyện, đường dẫn dự án và logger để ghi kết quả vào console lẫn file log.


Notebook nằm ở `src/notebooks/`, nên `ROOT` được xác định bằng thư mục gốc của dự án. Từ đó, thêm `src/` vào `sys.path` để import `models` và `train_utils`.

Logger có tên `"train"` để đồng bộ với `train_utils.py`. Khi chạy notebook, dữ liệu log được ghi đồng thời ra console và file `logs/train_<run_id>.log`; `run_id` theo thời gian giúp tránh ghi đè log cũ.


In [ ]:
import logging
import sys
import time
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents) if (path / "requirements.txt").exists())
sys.path.insert(0, str(ROOT / "src"))

from models import MODEL_NAMES, build_model, count_params
from train_utils import (
    FERDataset,
    balanced_sampler,
    fit,
    get_device,
    load_processed,
    set_seed,
    stratified_subset,
)

LOG_DIR = ROOT / "logs"
CKPT_DIR = ROOT / "exps" / "checkpoints"
HIST_DIR = ROOT / "exps" / "history"

LOG_DIR.mkdir(exist_ok=True)
run_id = time.strftime("%Y%m%d_%H%M%S")

logger = logging.getLogger("train")
logger.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s | %(name)s | %(levelname)s | %(message)s"
)

if not logger.handlers:
    console_handler = logging.StreamHandler()
    file_handler = logging.FileHandler(
        LOG_DIR / f"train_{run_id}.log",
        encoding="utf-8",
    )

    for handler in (console_handler, file_handler):
        handler.setFormatter(formatter)
        logger.addHandler(handler)

## **2. Cấu hình thí nghiệm**

Thiết lập kiến trúc, số epoch, batch size, learning rate, cơ chế dừng sớm, seed và chế độ chạy thử.


- **`arch`**: Kiến trúc cần huấn luyện. Mỗi kiến trúc có số tham số và thời gian/epoch khác nhau.
- **`epochs`**: Số vòng lặp qua toàn bộ tập train. Dùng 2 epoch cho chạy thử và 15 epoch cho lần huấn luyện thật.
- **`lr`**: Learning rate của Adam. Giá trị `1e-3` là mức mặc định an toàn.
- **`patience`**: Số epoch liên tiếp không cải thiện `val_loss` trước khi kích hoạt EarlyStopping.
- **`smoke`**: Khi `True`, chỉ dùng 512 mẫu train và 256 mẫu valid để kiểm tra pipeline, không dùng để đánh giá mô hình.
- **`seed`**: Seed cố định để kết quả có thể tái lập.


In [ ]:
CFG = {
    "arch": "medium",       # MODEL_NAMES: small, medium, medium_01, large
    "epochs": 2,
    "batch_size": 64,
    "lr": 1e-3,
    "patience": 3,
    "smoke": True,
    "seed": 42,
}

assert CFG["arch"] in MODEL_NAMES, (
    f"Kiến trúc không hợp lệ, chọn trong {MODEL_NAMES}"
)

set_seed(CFG["seed"])

## **3. Dữ liệu**

Nạp dữ liệu đã xử lý, lấy mẫu ngẫu nhiên khi chạy thử và tạo DataLoader cho hai tập train/valid.


`load_processed()` đọc ba tập dữ liệu train, valid, test cùng thống kê `mean/std` được tính trên tập train ở Ngày 1. `FERDataset` chuyển ảnh sang tensor, chia chuẩn hóa pixel và áp dụng augmentation cho tập train.

Khi `smoke=True`, lấy mẫu ngẫu nhiên thay vì cắt đầu danh sách. Cách lấy mẫu ngẫu nhiên giúp chạy thử có phân bố lớp đa dạng hơn.


In [ ]:
splits, stats = load_processed(ROOT / "data" / "processed")
x_tr, y_tr = splits["train"]
x_va, y_va = splits["valid"]

N_TRAIN = 512 if CFG["smoke"] else None
N_VALID = 256 if CFG["smoke"] else None

if N_TRAIN is not None:
    i_tr = stratified_subset(y_tr, N_TRAIN, CFG["seed"])
    x_tr, y_tr = x_tr[i_tr], y_tr[i_tr]
if N_VALID is not None:
    i_va = stratified_subset(y_va, N_VALID, CFG["seed"])
    x_va, y_va = x_va[i_va], y_va[i_va]

print("Phân bố nhãn train:", np.bincount(y_tr, minlength=7))

train_ds = FERDataset(x_tr, y_tr, stats["mean"], stats["std"], augment=True)
valid_ds = FERDataset(x_va, y_va, stats["mean"], stats["std"], augment=False)
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    sampler=balanced_sampler(y_tr),
    num_workers=0,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
)

device = get_device()
logger.info(
    "run=%s | train=%d | valid=%d | smoke=%s | cfg=%s",
    run_id, len(train_ds), len(valid_ds), CFG["smoke"], CFG,
)

## **4. Khởi tạo mô hình**

Tạo mô hình theo kiến trúc đã chọn trong cấu hình và chuyển mô hình sang thiết bị CPU hoặc GPU.


`build_model()` trả về một `EmotionCNN` theo cấu hình trong `models.py`. Mô hình xuất logits cho 7 lớp cảm xúc; hàm `fit()` sử dụng `CrossEntropyLoss` để tính loss.


In [ ]:
import os
print("Số nhân:", os.cpu_count(), "| luồng PyTorch đang dùng:", torch.get_num_threads())

torch.set_num_threads(os.cpu_count() // 2 if os.cpu_count() > 2 else 1)
print("Đặt lại luồng:", torch.get_num_threads())

In [ ]:
model = build_model(CFG["arch"]).to(device)

logger.info(
    "model=%s | trainable params=%.2fM",
    CFG["arch"],
    count_params(model) / 1e6,
)

## **5. Huấn luyện**

Gọi hàm `fit()` để huấn luyện mô hình, đánh giá trên tập valid, lưu checkpoint và ghi lịch sử từng epoch.


Hàm `fit()` thực hiện các bước sau mỗi epoch:

- Huấn luyện trên tập train và cập nhật trọng số, sau đó đánh giá trên tập valid.
- Khi `val_loss` giảm, lưu checkpoint tốt nhất vào `exps/checkpoints/`.
- Ghi loss, accuracy và thời gian vào file CSV trong `exps/history/`.
- Dừng sớm nếu `val_loss` không cải thiện trong `patience` epoch.

Quá trình huấn luyện cũng được ghi vào file log để theo dõi trực tiếp.


In [ ]:
ckpt_path, csv_path = fit(
    name=CFG["arch"],
    model=model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    device=device,
    epochs=CFG["epochs"],
    lr=CFG["lr"],
    run_id=run_id,
    logs_dir=HIST_DIR,
    ckpt_dir=CKPT_DIR,
    patience=CFG["patience"],
)

print("Checkpoint:", ckpt_path)
print("Lịch sử từng epoch:", csv_path)

## **6. Kiểm tra kết quả chạy thử**

Khi `smoke=True`, hãy kiểm tra các điều kiện sau trước khi chạy thí nghiệm thật:

1. Log có đủ dòng `epoch 1/2` và `epoch 2/2`, không có traceback.
2. `train_loss` giảm hoặc ổn định theo xu hướng hợp lý qua hai epoch.
3. Có file `.pt` trong `exps/checkpoints/` và file CSV trong `exps/history/`.

Nếu cả ba đều đúng, đặt `smoke=False`, `epochs=15` và chạy lại. Với kiến trúc `medium`, mỗi epoch trên CPU có thể mất khoảng 19 phút, do đó 15 epoch có thể cần khoảng 5 giờ.
